# Experiment 5: Few-Shot vs. Zero-Shot Comparison

Regenerated as a standalone notebook using Google Gemini API.

In [7]:

import os, json, re, getpass
try:
    from google import genai
except ImportError as exc:
    raise ImportError("Install the Gemini SDK first: %pip install -q google-genai") from exc

# Use a new Gemini API key. Input is hidden and not saved in this notebook.
API_KEY = os.getenv("GEMINI_API_KEY") or getpass.getpass("Enter your Gemini API key (input hidden): ")
MODEL = "gemini-2.5-flash"
client = genai.Client(api_key=API_KEY)

def ask_llm(prompt, system="You are a helpful assistant.", temperature=0.2):
    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config={"system_instruction": system, "temperature": temperature}
    )
    return response.text or ""

print("Gemini client configured. Model:", MODEL)


Enter your Gemini API key (input hidden): ··········
Gemini client configured. Model: gemini-2.5-flash


## Experiment Program

In [8]:
import time
from google.genai.errors import ClientError

reviews = [
 ("The battery life is amazing!", "Positive"),
 ("It works, nothing special.", "Neutral"),
 ("The camera is terrible and slow.", "Negative"),
 ("The screen is okay for the price.", "Neutral"),
 ("Best phone I have ever used!", "Positive")
]
examples = 'Examples: "Excellent product!" -> Positive; "It is average." -> Neutral; "Very poor quality." -> Negative.'

def ask_llm_with_retry(prompt, retries=5, delay=10):
    for i in range(retries):
        try:
            return ask_llm(prompt)
        except ClientError as e:
            if e.code == 429:
                print(f"Rate limit hit (429). Retrying in {delay} seconds...")
                time.sleep(delay)
                delay *= 2  # Exponential backoff
            else:
                raise e
    raise Exception("Failed after maximum retries due to rate limit.")

for review, truth in reviews:
    try:
        zero = ask_llm_with_retry(f"Classify this review as Positive, Neutral, or Negative. Output only the label. Review: {review}")
        # Optional delay between zero and few-shot call
        time.sleep(2)
        few = ask_llm_with_retry(f"{examples}\nClassify this review as Positive, Neutral, or Negative. Output only the label. Review: {review}")
        print({"review": review, "ground_truth": truth, "zero_shot": zero, "few_shot": few})
    except Exception as e:
        print(f"Could not process review: '{review}' due to error: {e}")

    # Wait 4 seconds between reviews to stay under the Requests Per Minute (RPM) threshold
    print("Waiting 4 seconds before the next review...")
    time.sleep(4)

{'review': 'The battery life is amazing!', 'ground_truth': 'Positive', 'zero_shot': 'Positive', 'few_shot': 'Positive'}
Waiting 4 seconds before the next review...
{'review': 'It works, nothing special.', 'ground_truth': 'Neutral', 'zero_shot': 'Neutral', 'few_shot': 'Neutral'}
Waiting 4 seconds before the next review...
{'review': 'The camera is terrible and slow.', 'ground_truth': 'Negative', 'zero_shot': 'Negative', 'few_shot': 'Negative'}
Waiting 4 seconds before the next review...
Rate limit hit (429). Retrying in 10 seconds...
Rate limit hit (429). Retrying in 20 seconds...
{'review': 'The screen is okay for the price.', 'ground_truth': 'Neutral', 'zero_shot': 'Neutral', 'few_shot': 'Neutral'}
Waiting 4 seconds before the next review...
{'review': 'Best phone I have ever used!', 'ground_truth': 'Positive', 'zero_shot': 'Positive', 'few_shot': 'Positive'}
Waiting 4 seconds before the next review...
